In [73]:
import torch
import torch.nn as nn
import torch.optim as optim
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# Load data
file_path = "../../dataStuff/UNSW_binData.csv"
data = pd.read_csv(file_path)

# Encode the target labels
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["label"])

# Split dataset into features (X) and target (y)
X = data.drop(columns=["label"])
y = data["label"]

# Split into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

# Initialize TabNetClassifier
model = TabNetClassifier(seed=SEED)

# Train the model in a single fit call
model.fit(
    X_train.values, y_train.values,
    eval_set=[(X_test.values, y_test.values)],
    max_epochs=50,  # Train for 50 epochs in a single call
    batch_size=1024,
    virtual_batch_size=128,
    patience=10
)

# Predictions
y_train_pred = model.predict(X_train.values)
y_val_pred = model.predict(X_test.values)

# Compute Metrics
train_loss = model.history.history.get("loss", [None])[-1]  # Get last train loss
train_acc = accuracy_score(y_train.values, y_train_pred)

# Find validation loss key dynamically
val_loss_key = next((key for key in model.history.history.keys() if "val" in key.lower()), None)
val_loss = model.history.history[val_loss_key][-1] if val_loss_key else None

val_acc = accuracy_score(y_test.values, y_val_pred)
val_prec = precision_score(y_test.values, y_val_pred, average='macro')
val_rec = recall_score(y_test.values, y_val_pred, average='macro')

# Print Final Metrics
print({
    "Train Loss": train_loss,
    "Train Accuracy": train_acc,
    "Val Loss": val_loss,
    "Val Accuracy": val_acc,
    "Val Precision": val_prec,
    "Val Recall": val_rec,
})

# Final Evaluation
print(f'\nFinal Accuracy: {val_acc}\n')
print(classification_report(y_test.values, y_val_pred))


c:\Users\Sid\AppData\Local\Programs\Python\Python311\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.10799 | val_0_auc: 0.98189 |  0:00:02s
epoch 1  | loss: 0.06089 | val_0_auc: 0.98904 |  0:00:04s
epoch 2  | loss: 0.059   | val_0_auc: 0.98965 |  0:00:06s
epoch 3  | loss: 0.05662 | val_0_auc: 0.98956 |  0:00:09s
epoch 4  | loss: 0.05823 | val_0_auc: 0.9951  |  0:00:11s
epoch 5  | loss: 0.05718 | val_0_auc: 0.99507 |  0:00:13s
epoch 6  | loss: 0.05589 | val_0_auc: 0.99609 |  0:00:15s
epoch 7  | loss: 0.05484 | val_0_auc: 0.99592 |  0:00:18s
epoch 8  | loss: 0.05508 | val_0_auc: 0.99302 |  0:00:20s
epoch 9  | loss: 0.05482 | val_0_auc: 0.989   |  0:00:22s
epoch 10 | loss: 0.05973 | val_0_auc: 0.9956  |  0:00:24s
epoch 11 | loss: 0.05565 | val_0_auc: 0.99623 |  0:00:26s
epoch 12 | loss: 0.0547  | val_0_auc: 0.99602 |  0:00:28s
epoch 13 | loss: 0.0547  | val_0_auc: 0.9952  |  0:00:30s
epoch 14 | loss: 0.05438 | val_0_auc: 0.99561 |  0:00:33s
epoch 15 | loss: 0.05353 | val_0_auc: 0.99592 |  0:00:35s
epoch 16 | loss: 0.05344 | val_0_auc: 0.99613 |  0:00:37s
epoch 17 | los

c:\Users\Sid\AppData\Local\Programs\Python\Python311\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


{'Train Loss': 0.05311544517439509, 'Train Accuracy': 0.9817210262096153, 'Val Loss': 0.9957596779812761, 'Val Accuracy': 0.9814598090545119, 'Val Precision': 0.9828562058347416, 'Val Recall': 0.9661285471423713}

Final Accuracy: 0.9814598090545119

              precision    recall  f1-score   support

           0       0.98      1.00      0.99     12337
           1       0.99      0.94      0.96      3898

    accuracy                           0.98     16235
   macro avg       0.98      0.97      0.97     16235
weighted avg       0.98      0.98      0.98     16235



In [76]:
# Get the expected number of features
num_features = X_train.shape[1]

# Sample input (Ensure it has the correct number of features)
sample_input = np.array([
    0.1, 20, 35, 1.5, 4.3, 10.2, 0, 1, 45, 0.67, 8, 3, 0.99, 0.5  # Added one more feature
])

# Check if sample_input matches expected feature count
if len(sample_input) != num_features:
    raise ValueError(f"Feature mismatch: Model expects {num_features} features, but received {len(sample_input)}")

# Reshape for model input
sample_input = sample_input.reshape(1, -1)

# Perform inference
predicted_label = model.predict(sample_input)
predicted_probabilities = model.predict_proba(sample_input)

# Display results
print(f"Predicted Label: {predicted_label[0]}")
print(f"Predicted Probabilities: {predicted_probabilities}")


Predicted Label: 0
Predicted Probabilities: [[1. 0.]]
